In [ ]:
import pandas as pd
import numpy as np
# from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import joblib

from IPython.display import Markdown, display

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, validation_curve
from sklearn.compose import make_column_selector
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.utils import resample

from statsmodels.formula.api import ols


# Explicación: Importante sobre la plantilla

Se tiene que tener en cuenta que este notebook no tiene ningún archivo asociado, ni se carga ningún DataFrame de prueba. Por lo tanto, es una plantilla que no debería ejecutarse como tal, es decir, de este apartado se debería poder identificar que código necesitan y usarlo en el notebook que entregarán en el parcial práctico.

Esta etapa del parcial es en la sala Waira. Deben utilizar los computadores de la sala y conectarse al usuario Examen. En los computadores está disponible Visual Studio Code con Python para editar y ejecutar los notebooks. 
Además, con el fin de que puedan resolver inquietudes de código, tendrán acceso a los siguientes sitios:

    https://scikit-learn.org/
    https://pandas.pydata.org/
    https://imbalanced-learn.org/stable/
    https://numpy.org/
    https://matplotlib.org/
    https://seaborn.pydata.org/
    https://pypi.org/

In [ ]:
df = pd.read_csv('tu_archivo.csv',sep=";", encoding='utf-8')

In [ ]:
df = pd.read_csv('tu_archivo.csv')

In [ ]:

# =========================================================
# Análisis exploratorio y calidad de datos
# =========================================================


In [ ]:
data = df.copy()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
df.describe(include=['object', 'category'])

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop(columns=['columna_1'], inplace=True)

In [ ]:
df.dropna(subset=["col1"])

In [ ]:
df.fillna(df['col2'].median(), inplace=True)

In [ ]:
df["columna"]

In [ ]:
df["Columna_nueva"] = 1

In [ ]:
df["Columna_nueva"] = df["col1"] * 2

In [ ]:
df['Vista'] = df['Vista'].abs()

In [ ]:
df['Vista'] = df['Vista'].astype(str).str[0].astype(int)

In [ ]:
df["Columna_nueva_2"] = np.where(df['col3'] > 1000000,1,0)

data.loc[data['FrenteMar'] == 'no', 'FrenteMar'] = 'NO'

In [ ]:
df[['col1', 'col2', 'col3', 'col4']]

In [ ]:
df['Columna_nueva_2'].value_counts()

In [ ]:
pd.crosstab(df['col1_cat'], df['col2_cat'])

In [ ]:
display(df['columna_categorica'].unique())
display(df['columna_categorica'].nunique())

In [ ]:
df[df['col1'] > 1000000]

df[df['col1'] == "valor_especifico"]

df.loc[df.loc[:,'col2']==795000620]

df[(df['col1'] < 0) | (data['col1'] > 4)].copy()

In [ ]:
data['Fecha'] = pd.to_datetime(data['Fecha'], format='%Y%m%dT%H%M%S')

data = data.sort_values(['Id', 'Fecha'], ascending=[True, False])

In [ ]:
encoder = OrdinalEncoder(categories=[['NO', 'SI']])

In [ ]:
df['FrenteMar'] = encoder.fit_transform(df[['FrenteMar']]).astype(int)

In [ ]:
df['variable_binaria'] = df['variable_texto'].map({'NO': 0, 'SI': 1})

In [ ]:

# =========================================================
# Transformaciones Previas (Fuera del Pipeline)
# =========================================================

df = df.drop_duplicates(subset=["Cuartos",'Baños'],keep='last')

In [ ]:
((df.isnull().sum()/df.shape[0])).sort_values(ascending=False)

In [ ]:
nulos = df[df['Calificación'].isna()].copy()

In [ ]:
indices = data.index[data['Calificación'].isna()]

In [ ]:
data.loc[indices]

In [ ]:
df_media = df.groupby('columna_categorica')['columna_numerica'].mean().reset_index()

df_agrupado = df.groupby('columna_categorica').agg({
    'columna_num1': 'mean',
    'columna_num2': ['sum', 'count', 'max']
}).reset_index()

df_conteo = df.groupby(['cat1', 'cat2']).size().reset_index(name='conteo')

In [ ]:
dup_counts = (data[["Id","Precio"]].groupby('Id').size().reset_index(name='Count').sort_values(by='Count', ascending=False))

dup_counts= dup_counts[dup_counts['Count']>1]

for id_, n in dup_counts[["Id","Count"]].values:    
    print(f"Id={id_} → {n} apariciones")

In [ ]:

dup_table = (
    data[data['Id'].duplicated(keep=False)]
    .copy()
    .assign(repeticiones=data.groupby('Id')['Id'].transform('size'))
    .sort_values(['repeticiones', 'Id'], ascending=[False, True])
)

In [ ]:

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
if 'target' in numeric_cols: numeric_cols.remove('target')

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[~((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR)))]

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()


In [ ]:
sns.heatmap(data.corr(numeric_only=True))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].scatter(data['Calificación'], data['AreaHabitable']); ax[0].set_title('Calificación vs AreaHabitable')
ax[1].scatter(data['Calificación'], data['AreaSuperior']); ax[1].set_title('Calificación vs AreaSuperior')
ax[2].scatter(data['Calificación'], data['AreaHabitable15']); ax[2].set_title('Calificación vs AreaHabitable15')

plt.tight_layout()
plt.show()

In [ ]:
df.hist(bins=30, figsize=(15, 10))
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(x=df['columna_numerica'])
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(x='columna_categorica', y='columna_numerica', data=df)
plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(x='columna_categorica', data=df)
plt.show()

plt.figure(figsize=(12, 8))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.show()

sns.pairplot(df, hue='target')
plt.show()

In [ ]:
sns.histplot(data=df, x='columna_numerica', kde=True, bins=30)
plt.show()

sns.boxplot(data=df, x='columna_categorica', y='columna_numerica')
plt.show()

sns.countplot(data=df, x='columna_categorica', order=df['columna_categorica'].value_counts().index)
plt.show()

sns.scatterplot(data=df, x='columna_num1', y='columna_num2', hue='columna_categorica')
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

In [ ]:
k = (data['Calificación'] / data['AreaHabitable']).dropna().median()


data= data.fillna({'Calificación': k * data['AreaHabitable']})
# Alternativa:
data.loc[data['Calificación'].isna(), 'Calificación'] = k * data['AreaHabitable']


data['Calificación'] = data['Calificación'].round()

data['Calificación'] = data['Calificación'].clip(lower=1, upper=13)

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_cols = encoder.fit_transform(df[['columna_categorica']])

df_encoded = pd.DataFrame(
    encoded_cols, 
    columns=encoder.get_feature_names_out(['columna_categorica'])
)

df = pd.concat([df.drop('columna_categorica', axis=1), df_encoded], axis=1)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target')
plt.show()

X = df.drop('target', axis=1)
y = df['target']

smote = SMOTE(random_state=42)
X_over, y_over = smote.fit_resample(X, y)

rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X, y)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=y_over, ax=axes[0])
sns.countplot(x=y_under, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Aplicación sobre los datos de entrenamiento
X_train_preparado = preprocessor.fit_transform(X_train)

# Visualización del resultado transformado
X_train_df_prep = pd.DataFrame(
    X_train_preparado, 
    columns=preprocessor.get_feature_names_out()
)
display(X_train_df_prep.head())